# PUF256 FPGA 驗證

獨立 PYNQ-Z2 overlay：640 RO / 320 候選配對 / 選 256 / 每組 5 次 / 80-bit helper。PUF、投票和 ECC 全部在 PL 執行；Python 只做 MMIO 與實驗紀錄。

載入此 overlay 會取代目前 PL 設計。請僅在已借用的板子執行，並確保沒有其他程式同時控制 PL。

此測試沒有 SHA3、身分認證、溫壓掃描或跨晶片唯一性驗證。

In [1]:
from pathlib import Path
import hashlib, json, platform
root = Path.cwd()
manifest = json.loads((root / 'MANIFEST.json').read_text())
for name, record in manifest['files'].items():
    if name.endswith('.ipynb'): continue  # Execution changes notebook outputs.
    data = (root / name).read_bytes()
    assert len(data) == record['bytes']
    assert hashlib.sha256(data).hexdigest() == record['sha256'], name
print('Upload hashes verified')
print('Python', platform.python_version())
print('Bitstream SHA256', manifest['files']['puf256.bit']['sha256'])


Upload hashes verified
Python 3.6.5
Bitstream SHA256 79346d7855c4a678563e8010083456b762b63bc8a8455388b2c45d4f12b7c7c2


## 完整實驗

以 32 輪 reference + 128 輪 training 篩選 320 組；根據訓練資料的零 fault/tie/flip/weak 與精確最小頻率差比例，選定並凍結 256 組。接著才做 32 輪獨立 holdout，不會因 holdout 失敗而替換配對。

硬體 Registration 只做一次；之後 100 次 Certification、一次共同 abort/reset 後重建、10 次重載相同 bitstream 後重建。重新載入時 bitmap/helper 寄存器會清空，由外部 enrollment.json 再提供。

CSV 包含實驗用 raw PUF 參考資料，僅供專題分析，不應當成正式產品的秘密儲存方案。整段可能需要數分鐘；每 8 輪列印進度。

In [2]:
from board_validation import run_validation
summary = run_validation(bitfile='puf256.bit', reference_reads=32, training_reads=128,
                         holdout_reads=32, reconstruction_runs=100, reload_runs=10)
assert summary['passed']


reference 8 / 32
reference 16 / 32
reference 24 / 32
reference 32 / 32
training 8 / 128
training 16 / 128
training 24 / 128
training 32 / 128
training 40 / 128
training 48 / 128
training 56 / 128
training 64 / 128
training 72 / 128
training 80 / 128
training 88 / 128
training 96 / 128
training 104 / 128
training 112 / 128
training 120 / 128
training 128 / 128
holdout 8 / 32
holdout 16 / 32
holdout 24 / 32
holdout 32 / 32
PUF reconstructions 10 / 100
PUF reconstructions 20 / 100
PUF reconstructions 30 / 100
PUF reconstructions 40 / 100
PUF reconstructions 50 / 100
PUF reconstructions 60 / 100
PUF reconstructions 70 / 100
PUF reconstructions 80 / 100
PUF reconstructions 90 / 100
PUF reconstructions 100 / 100
reload/reconstruct 1 / 10
reload/reconstruct 2 / 10
reload/reconstruct 3 / 10
reload/reconstruct 4 / 10
reload/reconstruct 5 / 10
reload/reconstruct 6 / 10
reload/reconstruct 7 / 10
reload/reconstruct 8 / 10
reload/reconstruct 9 / 10
reload/reconstruct 10 / 10
{
  "passed": true,
  "

## 檢查並匯出結果

每次執行會產生獨立 results_日期時間 資料夾。summary.json 只有全部通過才會產生 passed=true；中斷或錯誤時看 progress.json 與既有 CSV，不能宣稱完成。將該資料夾下載後，以 00_TESTBED/verify_board_results.py 獨立重算篩選、helper 與重建結果。

In [3]:
folders = sorted(p for p in root.glob('results_*') if p.is_dir())
latest = folders[-1]
print('Latest results:', latest.name)
print((latest / 'summary.json').read_text())
import zipfile
archive = root / (latest.name + '.zip')
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(latest.iterdir()):
        if path.is_file(): bundle.write(str(path), latest.name + '/' + path.name)
from IPython.display import FileLink, display
display(FileLink(archive.name))


Latest results: results_20261112_181403
{
  "bitstream_sha256": "79346d7855c4a678563e8010083456b762b63bc8a8455388b2c45d4f12b7c7c2",
  "elapsed_seconds": 284.7119544772431,
  "eligible_pairs": 288,
  "holdout": {
    "failed_pairs": [],
    "passed": true,
    "reads_per_pair": 32,
    "selected_count": 256
  },
  "holdout_samples": 8192,
  "passed": true,
  "reconstruction_count": 111,
  "reference_samples": 10240,
  "reload_count": 10,
  "scope": "Single board, fixed routed bitstream, ambient conditions. No authentication, SHA3, entropy or multi-device uniqueness claim.",
  "selected_pairs": 256,
  "total_corrected_words": 0,
  "training_samples": 40960
}


/home/xilinx/jupyter_notebooks/puf256_a0981_20261113/results_20261112_181403.zip